# Reverse Engineering: Dräger PulmoVista 500 `.eit` Format

**Format version**: 51 (PulmoVista 500)  
**Files used**: `patient01.eit` + `patient02.eit` (V1.30, 50 Hz); `HealthyLung.eit` (V1.00, 20 Hz) in frame-count table  

## Attribution and methodology

The frame internal layout and calibration formulas come from studying **EIDORS**
(`eidors_readdata.m`, function `read_draeger_file`, A. Adler et al., GPL-3.0).
Only the data format description has beenused to understand field offsets and calibration constants.
All values are independently verified against the real files in this notebook.
The calibration factors `ft = [0.00098242, 0.00019607]` are empirical constants
from EIDORS, not documented in the source.

## Prior art

- **EIDORS** (GPL-3.0, MATLAB): `eidors/interface/eidors_readdata.m` — subfunctions
  `read_draeger_header` and `read_draeger_file` (after line 1402). Used to understand
  field layout and calibration. (GPL/Apache-2.0 incompatible).  
  https://eidors3d.sourceforge.net/doc/index.html?eidors/interface/eidors_readdata.html


In [1]:
import struct
import numpy as np
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'fasteit').exists():
            return candidate
    raise FileNotFoundError('Could not locate fastEIT repo root')

REPO_ROOT  = find_repo_root(Path.cwd())
TEST_FILES = REPO_ROOT / 'src' / 'fasteit' / 'test_files'
FRAME_SIZE = 5495

EIT_FILES = {
    'HealthyLung': TEST_FILES / 'HealthyLung.eit',
    'patient01':   TEST_FILES / 'patient01.eit',
    'patient02':   TEST_FILES / 'patient02.eit',
}

for name, path in EIT_FILES.items():
    print(f'{path.stat().st_size/1e6:7.2f} MB  {path.name}')

   6.15 MB  HealthyLung.eit
  63.20 MB  patient01.eit
  66.77 MB  patient02.eit


## 1. File header

The `.eit` file begins with a **12-byte binary preamble**, then an **ASCII text block**,
then an **8-byte separator**, then the raw binary frames.

```
offset 0         : preamble  (12 bytes, 3 × int32 LE)
offset 12        : ASCII header text  (latin-1, variable length)
offset sep_offset: separator  b'**\r\n\r\n\r\n'  (8 bytes)
offset sep_offset+8: binary frames begin
```

### 1.1 Preamble (bytes 0-11)

The first 12 bytes are a binary preamble: **3 × int32 LE**.

| bytes | field | value (patient01) |
|---|---|---|
| [0:4] | `format_version` | 51 |
| [4:8] | `sep_offset` | offset of the separator `b'**\r\n\r\n\r\n'` |
| [8:12] | `unknown` | unknown int32 |


In [2]:
def xxd(data, base_offset=0, annotations=None):
    """xxd-style hexdump with optional annotations at specific offsets."""
    lines = []
    for row in range(0, len(data), 16):
        off = base_offset + row
        if annotations and off in annotations:
            lines.append(f'  # {annotations[off]}')
        chunk = data[row:row + 16]
        hex_p = ' '.join(f'{b:02x}' for b in chunk)
        asc_p = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
        lines.append(f'{off:08x}: {hex_p:<47}  |{asc_p}|')
    return '\n'.join(lines)

raw = EIT_FILES['patient01'].read_bytes()
v0, v1, v2 = struct.unpack('<3i', raw[:12])
print('=== patient01.eit — preamble (bytes 0–11) ===')
print(xxd(raw[:12], annotations={
    0: 'format_version (int32 LE)',
    4: 'sep_offset     (int32 LE)',
    8: 'unknown        (int32 LE)',
}))
print(f'\nformat_version = {v0}   sep_offset = {v1}   unknown = {v2}')


=== patient01.eit — preamble (bytes 0–11) ===
  # format_version (int32 LE)
00000000: 33 00 00 00 7a 1d 00 00 2a 06 00 00              |3...z...*...|

format_version = 51   sep_offset = 7546   unknown = 1578


### 1.2 ASCII header 

The ASCII header runs from byte 12 to `sep_offset − 1`. It contains the System Data section
with device parameters, firmware version, framerate, and acquisition settings.

The separator `b'**\r\n\r\n\r\n'` (8 bytes) immediately follows at `sep_offset`.
Binary frames begin at `sep_offset + 8`.


In [3]:
def parse_header(raw):
    v0, v1, v2 = struct.unpack('<3i', raw[:12])
    text = raw[12:v1].decode('latin-1')
    fields = {}
    for line in text.split('\r\n'):
        if ':' in line:
            k, _, v = line.partition(':')
            fields[k.strip()] = v.strip()
    return v0, v1, v1 + 8, fields

_SENSITIVE = {'Filename', 'Name', 'Age / Date of Birth', 'Patient ID', 'Short Comment'}

def redact_header(text):
    """Replace values of sensitive Patient Data fields with [REDACTED]."""
    lines = []
    for line in text.split('\r\n'):
        colon = line.find(':')
        if colon != -1 and line[:colon].strip() in _SENSITIVE:
            lines.append(line[:colon + 1] + ' [REDACTED]')
        else:
            lines.append(line)
    return '\r\n'.join(lines)

raw = EIT_FILES['patient01'].read_bytes()
_, sep_off, bin_start, _ = parse_header(raw)

print('=== patient01.eit — ASCII header (bytes 12 to sep_offset) ===')
print(redact_header(raw[12:sep_off].decode('latin-1')))

print(f'=== separator at offset {sep_off} + start of frame 0 ===')
print(xxd(raw[sep_off:sep_off + 24], base_offset=sep_off, annotations={
    sep_off:     "separator b'**\\r\\n\\r\\n\\r\\n' (8 bytes)",
    sep_off + 8: 'frame 0 byte 0: float64 timestamp begins here',
}))


=== patient01.eit — ASCII header (bytes 12 to sep_offset) ===


*******************************
---Draeger EIT-Software V1.30---

Date:      04.01.2024
Time:      18:10:54.015
Filename: [REDACTED]
Format:    51

*******************************
---------Patient Data----------

Name                : [REDACTED]
Age / Date of Birth : [REDACTED]
Patient ID          : [REDACTED]
Short Comment       : [REDACTED]
Notes               :                                                   

*******************************
-----DataAnalysis Settings-----

....................................................................................................................................................................................................................................................................................................................................................................................................................................................................

## 2. Frame size proof

If the binary section is a flat sequence of equal-size frames,
then `(file_size − binary_start) / frame_size` must be an exact integer.

Test the candidate 5495 bytes/frame on all three files.

In [4]:
FRAME_SIZE = 5495
print(f'FRAME_SIZE = {FRAME_SIZE} bytes\n')

framerates = {'HealthyLung': 20, 'patient01': 50, 'patient02': 50}
for name, path in EIT_FILES.items():
    raw = path.read_bytes()
    _, sep_off, bin_start, _ = parse_header(raw)
    binary_bytes = len(raw) - bin_start
    n = binary_bytes / FRAME_SIZE
    fr = framerates[name]
    print(f'  {path.name:<18}: ({len(raw)} - {bin_start}) / {FRAME_SIZE} = {n} frames  ({n/fr:.1f} s @ {fr} Hz)')


FRAME_SIZE = 5495 bytes

  HealthyLung.eit   : (6150823 - 7413) / 5495 = 1118.0 frames  (55.9 s @ 20 Hz)
  patient01.eit     : (63200054 - 7554) / 5495 = 11500.0 frames  (230.0 s @ 50 Hz)
  patient02.eit     : (66771796 - 7546) / 5495 = 12150.0 frames  (243.0 s @ 50 Hz)


## 3. Frame 0 — annotated hexdump

Each frame is 5495 bytes. Showing selected sections with field annotations.
Full frame = 344 rows × 16 bytes.

**Layout summary** (bytes within each frame, 0-indexed):

| offset (dec) | offset (hex) | bytes | type | field |
|---|---|---|---|---|
| 0 | 0x0000 | 8 | float64 LE | `timestamp` (fraction of day) |
| 8 | 0x0008 | 8 | float64 LE | `unknown` (value ~64–89, varies per file) |
| 16 | 0x0010 | 1664 | float64[208] LE | `trans_A` — 208 raw transimpedance values (Set A) |
| 1680 | 0x0690 | 128 | float64[16] LE | `unknown_16a` |
| 1808 | 0x0710 | 128 | float64[16] LE | `injection_current` (÷ 194326.3536 → Ampere) |
| 1936 | 0x0790 | 128 | float64[16] LE | `unknown_16b` |
| 2064 | 0x0810 | 128 | float64[16] LE | `voltage_A` |
| 2192 | 0x0890 | 400 | float64[50] LE | `unknown_50` |
| 2592 | 0x0A20 | 1664 | float64[208] LE | `trans_B` — 208 raw values (Set B, reference) |
| 4256 | 0x10A0 | 384 | float64[48] LE | `unknown_48` |
| 4640 | 0x1220 | 128 | float64[16] LE | `voltage_B` |
| 4768 | 0x12C0 | 48 | float64[6] LE | `unknown_6` (end of 600-double block) |
| 4816 | 0x12D0 | 352 | float64[44] LE | `gugus` — 44 values, **ignored by EIDORS** |
| 5168 | 0x1430 | 1 | uint8 | `unknown_byte` (value 0xFF in both files) |
| 5169 | 0x1431 | 268 | float32[67] LE | `medibus` — 67 ventilator channels |
| 5437 | 0x153D | 30 | bytes | `event_text` — ASCII, space-padded |
| 5467 | 0x155B | 24 | bytes | `mixed` — int16 + zeros (EIDORS: uninterpreted) |
| 5491 | 0x1573 | 2 | uint16 LE | `frame_counter` |
| 5493 | 0x1575 | 2 | uint8[2] | `padding` (zeros) |
| **5495** | | | | end of frame |



In [5]:
raw = EIT_FILES['patient01'].read_bytes()
_, v1, bin_start, _ = parse_header(raw)
bin_start = v1 + 8
frame0 = raw[bin_start: bin_start + FRAME_SIZE]

# ── Annotated frame hexdump: show selected sections ──────────────────────
FRAME_ANN = {
    0x0000: '[0x0000] ts          float64 LE  timestamp (fraction of day)',
    0x0010: '[0x0010] trans_A[0]  float64 LE  first of 208 raw transimpedance values (Set A)',
    0x0690: '[0x0690] end of trans_A[0..207]; unknown_16a begins',
    0x0710: '[0x0710] injection_current[0..15]  float64 LE  raw / 194326.3536 = Ampere',
    0x10a0: '[0x10a0] end of trans_B[0..207]; unknown_48 begins',
    0x1220: '[0x1220] voltage_B[0..15]  float64 LE',
    0x12c0: '[0x12c0] unknown_6 — last 6 doubles of 600-double signal block',
    0x12d0: '[0x12d0] gugus — 44 float64 LE, ignored by EIDORS (purpose unknown)',
    0x1430: '[0x1430] unknown_byte = 0xFF (1 byte)',
    0x1431: '[0x1431] medibus[0..66] — 67 x float32 LE (first 4: -3.4e38 = not connected)',
    0x153d: '[0x153d] event_text (30 bytes, space-padded ASCII)',
    0x1573: '[0x1573] frame_counter  uint16 LE',
}

SECTIONS = [
    (0x0000, 0x0030, 'timestamp + unknown + trans_A start'),
    (0x0680, 0x0720, 'trans_A end + unknown_16 + injection_current start'),
    (0x10a0, 0x1260, 'trans_B end + voltage_B'),
    (0x12c0, 0x1440, 'end of signal block + gugus'),
    (0x1430, 0x154a, 'unknown_byte + medibus + tail'),
]

print('patient01.eit — frame 0 (5495 bytes), selected sections')
for start, end, title in SECTIONS:
    sec_ann = {k: v for k, v in FRAME_ANN.items() if start <= k < end}
    print(f'\n=== [0x{start:04x}..0x{end-1:04x}] {title} ===')
    print(xxd(frame0[start:end], start, sec_ann))
    if start == 0x0000:
        ts  = struct.unpack('<d', frame0[0:8])[0]
        unk = struct.unpack('<d', frame0[8:16])[0]
        ta0 = struct.unpack('<d', frame0[16:24])[0]
        ta1 = struct.unpack('<d', frame0[24:32])[0]
        ta2 = struct.unpack('<d', frame0[32:40])[0]
        h,m,s = int(ts*24), int(ts*24*60)%60, ts*86400%60
        print(f'\n  [0:8]   float64 = {ts:.10f}  ->  {h:02d}:{m:02d}:{s:02.0f}  (header Time: 06:10:03 PM = 18:10)')
        print(f'  [8:16]  float64 = {unk:.6f}     ->  unknown field (patient01: 89.080)')
        print(f'  [16:24] float64 = {ta0:.6f}     ->  trans_A[0]  ~{ta0:.1f} Ω')
        print(f'  [24:32] float64 = {ta1:.6f}      ->  trans_A[1]  ~{ta1:.1f} Ω')
        print(f'  [32:40] float64 = {ta2:.6f}      ->  trans_A[2]  ~{ta2:.1f} Ω')

patient01.eit — frame 0 (5495 bytes), selected sections

=== [0x0000..0x002f] timestamp + unknown + trans_A start ===
  # [0x0000] ts          float64 LE  timestamp (fraction of day)
00000000: b5 d6 3e cf 02 3e e8 3f b3 0f 31 2f 21 45 56 40  |..>..>.?..1/!EV@|
  # [0x0010] trans_A[0]  float64 LE  first of 208 raw transimpedance values (Set A)
00000010: 4d 72 46 d4 c4 f7 27 40 8f 64 e8 a5 4a 61 12 40  |MrF...'@.d..Ja.@|
00000020: c1 0d ad 25 15 8f 07 40 40 63 88 07 fd 00 fd 3f  |...%...@@c.....?|

  [0:8]   float64 = 0.7575696991  ->  18:10:54  (header Time: 06:10:03 PM = 18:10)
  [8:16]  float64 = 89.080150     ->  unknown field (patient01: 89.080)
  [16:24] float64 = 11.983924     ->  trans_A[0]  ~12.0 Ω
  [24:32] float64 = 4.595011      ->  trans_A[1]  ~4.6 Ω
  [32:40] float64 = 2.944865      ->  trans_A[2]  ~2.9 Ω

=== [0x0680..0x071f] trans_A end + unknown_16 + injection_current start ===
00000680: ce f3 05 5c db b7 52 40 1c d4 0c 7a 74 51 46 bf  |...\..R@...ztQF.|
  # [0x0690] end

## 4. Transimpedance measurements

**The values are `float64` (8 bytes each).** The confirmed sequence starting at byte 16.

### transimpedance_A and transimpedance_B

The EIDORS code reads two sets of 208 values each (Set A and Set B).
They represent two separate voltage measurements made during the same frame.
The calibrated transimpedance is derived by combining both sets
(see calibration formula below).

In [6]:
raw = EIT_FILES['patient01'].read_bytes()
_, sep_off, _, _ = parse_header(raw)
bin_start = sep_off + 8
frame0 = raw[bin_start: bin_start + FRAME_SIZE]

trans_a = np.frombuffer(frame0[16:16 + 208 * 8], dtype='<f8')
trans_b = np.frombuffer(frame0[2592:2592 + 208 * 8], dtype='<f8')

print('=== patient01.eit — frame 0: transimpedance_A and transimpedance_B ===\n')
for label, arr, span in [('transimpedance_A', trans_a, 'frame bytes [16:1680]'),
                          ('transimpedance_B', trans_b, 'frame bytes [2592:4256]')]:
    print(f'{label}  (208 float64, {span}):')
    print(f'  count : {len(arr)}')
    print(f'  min   : {arr.min():+.6f}')
    print(f'  max   : {arr.max():+.6f}')
    print(f'  mean  : {arr.mean():+.6f}')
    print()


=== patient01.eit — frame 0: transimpedance_A and transimpedance_B ===

transimpedance_A  (208 float64, frame bytes [16:1680]):
  count : 208
  min   : -0.000681
  max   : +156.971689
  mean  : +12.274894

transimpedance_B  (208 float64, frame bytes [2592:4256]):
  count : 208
  min   : -25.040808
  max   : +0.000078
  mean  : -2.348294



## 6. Calibration formula

### Source and limitations

The formula below comes from the EIDORS source code, function `read_draeger_file`,
commented as authored by A. Adler, 2016-04-07. The two scale factors
`ft = [0.00098242, 0.00019607]` are empirical hardware calibration constants
specific to the PulmoVista 500.

The formula combines Set A (primary measurement) and Set B (reference measurement)
to produce the calibrated transimpedance voltage `vv` used for image reconstruction:

```python
ft = [0.00098242, 0.00019607]
vv[i]   = ft[0] * trans_A[i] - ft[1] * trans_B[i]   # calibrated transimpedance
curr[j] = injection_current[j] / 194326.3536          # electrode current in Ampere
volt[j] = (voltage_A[j] - voltage_B[j]) / 0.11771    # electrode voltage
```

In [7]:
frame0 = raw[bin_start: bin_start + FRAME_SIZE]

ft = [0.00098242, 0.00019607]
trans_a = np.frombuffer(frame0[16:16+208*8], dtype='<f8')
trans_b = np.frombuffer(frame0[2592:2592+208*8], dtype='<f8')
vv      = ft[0] * trans_a - ft[1] * trans_b

inj_raw  = np.frombuffer(frame0[1808:1808+16*8], dtype='<f8')
inj_a    = inj_raw / 194326.3536

print(f'=== patient01.eit — calibrated vv = ft[0]*A - ft[1]*B ===')
print(f'ft = {ft}\n')
print('vv (208 values):')
print(f'  min    : {vv.min():9.6f}  (Ω or V, unit not confirmed)')
print(f'  max    : {vv.max():9.6f}')
print(f'  mean   : {vv.mean():9.6f}')
print(f'  any NaN: {np.isnan(vv).any()}')
print(f'  any Inf: {np.isinf(vv).any()}')



=== patient01.eit — calibrated vv = ft[0]*A - ft[1]*B ===
ft = [0.00098242, 0.00019607]

vv (208 values):
  min    : -0.000001  (Ω or V, unit not confirmed)
  max    :  0.159102
  mean   :  0.012520
  any NaN: False
  any Inf: False


## 7. Medibus (67 float32)

The `.eit` frame contains **67 float32 ventilator channels** at offset 5169
(vs 52 or 58 in `.bin` — different channel count, different set).

Same two sentinels as in `.bin`:

| hex | float32 | meaning |
|---|---|---|
| `0xFF7FC99E` | ≈ −3.4×10³⁸ | MEDIBUS hardware **not connected** |
| `0xC47A0000` | −1000.0 | connected, **value not yet calculated** |

In [8]:
MEDIBUS_OFFSET = 5169
MEDIBUS_COUNT  = 67

for label, path in [('patient01.eit', EIT_FILES['patient01']),
                     ('patient02.eit', EIT_FILES['patient02'])]:
    raw = path.read_bytes()
    _, sep_off, _, _ = parse_header(raw)
    f0_start = sep_off + 8
    f0 = raw[f0_start: f0_start + FRAME_SIZE]
    med = np.frombuffer(f0[MEDIBUS_OFFSET: MEDIBUS_OFFSET + MEDIBUS_COUNT*4], dtype='<f4')

    large_neg = (med < -1e30).sum()
    exactly_neg1000 = (med == -1000.0).sum()
    other = med[(med >= -1e30) & (med != -1000.0)]

    print(f'=== {label} — medibus frame 0 (67 float32 at offset {MEDIBUS_OFFSET}) ===')
    print(f'count          : {len(med)}')
    print(f'large-neg (<-1e30) : {large_neg}   (0xFF7FC99E ≈ -3.4e38, not connected)')
    print(f'exactly -1000  : {exactly_neg1000:2d}   (0xC47A0000, no data yet)')
    print(f'other values   : {len(other):2d}   [{"  ".join(f"{v:.4f}" for v in other[:5])}]')
    if label == 'patient01.eit':
        print()
        print(f'idx 0..3  -> -3.4e38  (first 4 channels: MEDIBUS hardware not connected)')
        print(f'idx 4..63 -> -1000.0  (connected but no data calculated for these channels)')
        print(f'idx 64..66-> ~0')
    print()

=== patient01.eit — medibus frame 0 (67 float32 at offset 5169) ===
count          : 67
large-neg (<-1e30) : 4   (0xFF7FC99E ≈ -3.4e38, not connected)
exactly -1000  : 60   (0xC47A0000, no data yet)
other values   :  3   [0.0000  0.0000  0.0000]

idx 0..3  -> -3.4e38  (first 4 channels: MEDIBUS hardware not connected)
idx 4..63 -> -1000.0  (connected but no data calculated for these channels)
idx 64..66-> ~0

=== patient02.eit — medibus frame 0 (67 float32 at offset 5169) ===
count          : 67
large-neg (<-1e30) : 4   (0xFF7FC99E ≈ -3.4e38, not connected)
exactly -1000  : 60   (0xC47A0000, no data yet)
other values   :  3   [0.0000  0.0000  0.0000]



## 8. Frame tail — event text, frame counter, padding

In [9]:
raw = EIT_FILES['patient01'].read_bytes()
_, sep_off, _, _ = parse_header(raw)
bin_start = sep_off + 8
frame0 = raw[bin_start: bin_start + FRAME_SIZE]

ubyte   = frame0[5168]
ev_text = frame0[5437:5467]
counter = struct.unpack('<H', frame0[5491:5493])[0]
padding = frame0[5493:5495]

## 9. `.eit` vs `.bin` — timestamp cross-check

The `.eit` and `.bin` files for the same patient use the same timestamp encoding.
Frame 0 of both files should have the same (or very close) timestamp.

In [10]:
from fasteit.parsers.draeger.bin.draeger_dtypes import FRAME_EXT_DTYPE

p01_eit = EIT_FILES['patient01'].read_bytes()
_, sep_off, _, _ = parse_header(p01_eit)
bin_start_eit = sep_off + 8
ts_eit = struct.unpack('<d', p01_eit[bin_start_eit: bin_start_eit+8])[0]

bin_data = np.memmap(TEST_FILES / 'patient01.bin', dtype=FRAME_EXT_DTYPE, mode='r')
ts_bin   = float(bin_data[0]['ts'])

h_e, m_e = int(ts_eit*24), int(ts_eit*24*60)%60; s_e = ts_eit*86400%60
h_b, m_b = int(ts_bin*24), int(ts_bin*24*60)%60; s_b = ts_bin*86400%60

print(f'.eit frame 0: {ts_eit:.10f}  ({h_e:02d}:{m_e:02d}:{s_e:06.3f})')
print(f'.bin frame 0: {ts_bin:.10f}  ({h_b:02d}:{m_b:06.3f})')
print(f'Delta        : {abs(ts_eit-ts_bin)*86400*1000:.3f} ms')
print()
n_eit = len(p01_eit[bin_start_eit:]) // FRAME_SIZE
print(f'.eit n_frames: {n_eit}  ({n_eit/50:.1f} s)')
print(f'.bin n_frames: {len(bin_data)}  ({len(bin_data)/50:.1f} s)')
print()
print('Identical timestamps — same recording, same time base.')

.eit frame 0: 0.7575696991  (18:10:54.022)
.bin frame 0: 0.7575696991  (18:10.000)
Delta        : 0.000 ms

.eit n_frames: 11500  (230.0 s)
.bin n_frames: 11500  (230.0 s)

Identical timestamps — same recording, same time base.


## Summary

### File structure

| offset | bytes | type | field |
|---|---|---|---|
| 0 | 4 | int32 LE | `format_version` = 51 |
| 4 | 4 | int32 LE | `separator_offset` (byte offset of `**\r\n\r\n\r\n`) |
| 8 | 4 | int32 LE | `unknown_int` (varies per file, purpose not confirmed) |
| 12 | `sep_off − 12` | latin-1 text | ASCII header (device + patient metadata) |
| `sep_off` | 8 | bytes | separator `b'**\r\n\r\n\r\n'` |
| `sep_off + 8` | N × 5495 | binary | raw EIT frames |

### Per-frame layout (5495 bytes)

| offset (dec) | offset (hex) | bytes | type | field | status |
|---|---|---|---|---|---|
| 0 | 0x0000 | 8 | float64 LE | `timestamp` | **confirmed** |
| 8 | 0x0008 | 8 | float64 LE | `unknown` (~64–89) | ambiguous |
| 16 | 0x0010 | 1664 | float64[208] | `trans_A` | **confirmed** |
| 1680 | 0x0690 | 128 | float64[16] | `unknown_16a` | from EIDORS |
| 1808 | 0x0710 | 128 | float64[16] | `injection_current` | from EIDORS |
| 1936 | 0x0790 | 128 | float64[16] | `unknown_16b` | from EIDORS |
| 2064 | 0x0810 | 128 | float64[16] | `voltage_A` | from EIDORS |
| 2192 | 0x0890 | 400 | float64[50] | `unknown_50` | from EIDORS |
| 2592 | 0x0A20 | 1664 | float64[208] | `trans_B` | **confirmed** |
| 4256 | 0x10A0 | 384 | float64[48] | `unknown_48` | from EIDORS |
| 4640 | 0x1220 | 128 | float64[16] | `voltage_B` | from EIDORS |
| 4768 | 0x12C0 | 48 | float64[6] | `unknown_6` | from EIDORS |
| 4816 | 0x12D0 | 352 | float64[44] | `gugus` (EIDORS name) | from EIDORS |
| 5168 | 0x1430 | 1 | uint8 | `unknown_byte` (= 0xFF) | **confirmed** |
| 5169 | 0x1431 | 268 | float32[67] | `medibus` | **confirmed** |
| 5437 | 0x153D | 30 | bytes | `event_text` | **confirmed** |
| 5467 | 0x155B | 24 | bytes | `mixed` (int16+zeros) | from EIDORS |
| 5491 | 0x1573 | 2 | uint16 | `frame_counter` | **confirmed** |
| 5493 | 0x1575 | 2 | bytes | `padding` | **confirmed** |


### Calibration (EIDORS-sourced, A. Adler 2016)

```python
ft = [0.00098242, 0.00019607]         # empirical hardware constants, PulmoVista 500
vv[i]   = ft[0] * trans_A[i] - ft[1] * trans_B[i]   # calibrated transimpedance
curr[j] = injection_current[j] / 194326.3536          # injection current in Ampere
volt[j] = (voltage_A[j] - voltage_B[j]) / 0.11771    # electrode voltage
```

### Medibus sentinels (same as `.bin`)

| hex | float32 | meaning |
|---|---|---|
| `0xFF7FC99E` | ≈ −3.4×10³⁸ | not connected |
| `0xC47A0000` | −1000.0 | connected, no data |

## Open questions

1. **Byte [8:16]** (`unknown`): value ~64 (HealthyLung) vs ~89 (patient01).
   Could be `trans_A[0]` (injection-pair self-impedance) or a separate header
   field. Needs verification against EIDORS reconstructed output .

2. **`gugus` block [4816:5168]** (44 float64): EIDORS explicitly ignores these.
   Physical meaning unknown.

3. **`unknown_byte` [5168]** = 0xFF in all frames checked: consistent with
   padding or an unused status field.

4. **Calibration factors `ft`**: empirical constants from EIDORS,
   derivation not documented.

5. **Medibus channel names** (67 float32 in `.eit` vs 52/58 in `.bin`):
   mapping not yet established. 